<a href="https://colab.research.google.com/github/Tulasisree/AI_NOTES/blob/main/Deep_Q_Learning_for_Lunar_Landing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Learning for Lunar Landing

## Part 0 - Installing the required packages and importing the libraries

### Installing Gymnasium

In [ ]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!apt-get install -y swig
!pip install gymnasium[box2d]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  swig4.0
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  swig swig4.0
0 upgraded, 2 newly installed, 0 to remove and 3 not upgraded.
Need to get 1,116 kB of archives.
After this operation, 5,542 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig4.0 amd64 4.0.2-1ubuntu1 [1,110 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig all 4.0.2-1ubuntu1 [5,632 B]
Fetched 1,116 kB in 0s (4,933 kB/s)
Selecting previously unselected package swig4.0.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubun

### Importing the libraries

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.autograd as autograd
from torch.autograd import Variable
from collections import deque, namedtuple

## Part 1 - Building the AI

### Creating the architecture of the Neural Network

In [ ]:
class Network(nn.Module):
  #BRAIN
  def __init__(self, state_size, action_size, seed = 42):
      super(Network,self).__init__()
      self.seed = torch.manual_seed(seed)
      #first fully connected layer no. of neurons = 64, after experimentation this was decided for lunar landing
      self.fc1 = nn.Linear(state_size,64)
      #if we want only 1 fc, the second arg will be the no. of neurons in o/p layer
      #btr architecture contains fc2
      self.fc2 = nn.Linear(64,64)
      self.fc3 = nn.Linear(64,action_size)

  #forward method for neural nets
  #state is taken as the arg coz its gng to propogate the signal from state to o/p layer in order to play a certain action at a acertain time stamp.
  def forward(self,state):
      #1st we are propogating state to the 1st fc1
      x = self.fc1(state)
      #rectifier activation function
      x = F.relu(x)
      # to fc2
      x = self.fc2(x)
      x = F.relu(x)
      return self.fc3(x)


## Part 2 - Training the AI

### Setting up the environment

In [ ]:
import gymnasium as gym
#env on whcih our model will be trained
env = gym.make("LunarLander-v3")
state_shape = env.observation_space.shape
#1st index of shape contains the size
state_size = env.observation_space.shape[0]
number_actions = env.action_space.n
print('state_shape: ', state_shape)
print('state_size: ', state_size)
print('number_actions: ', number_actions)

state_shape:  (8,)
state_size:  8
number_actions:  4


### Initializing the hyperparameters

In [ ]:
learning_rate = 5e-4 #after experimentation this value was decided
minibatch_size = 100
discount_factor = 0.99
replay_buffer_size = int(1e5) #10000
interpolation_parameter = 1e-3

### Implementing Experience Replay

In [ ]:
#usually this is the name we give to class that implements experience replay
class ReplayMemory(object):
  #capacity of memory
  def __init__(self,capacity):
      #if we want to use GPU
      self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
      self.capacity = capacity #max size of memeory buffer
      self.memory = [] #list that stores the experiences,reward,nextstate and whether we are done or not

  #adds experience into memeory buffer
  def push(self, event):
    self.memory.append(event)
    if len(self.memory)>self.capacity:
      del self.memory[0]

  #sample to learn on
  def sample(self,batch_size):
    experiences = random.sample(self.memory, k = batch_size)
    #create stack of states using vstack()
    #state is in index 0 of each experience
    #convert these states to pytorch tensors, data type accepted is float
    #to() moving the stack of states to designated device
    states = torch.from_numpy(np.vstack([e[0] for e in experiences if e is not None])).float().to(self.device)
    actions = torch.from_numpy(np.vstack([e[1] for e in experiences if e is not None])).long().to(self.device)
    rewards = torch.from_numpy(np.vstack([e[2] for e in experiences if e is not None])).float().to(self.device)
    next_states = torch.from_numpy(np.vstack([e[3] for e in experiences if e is not None])).float().to(self.device)
    dones = torch.from_numpy(np.vstack([e[4] for e in experiences if e is not None]).astype(np.uint8)).float().to(self.device)
    return states,next_states,actions,rewards,dones

### Implementing the DQN class

In [ ]:
class Agent():
  def __init__(self,state_size,action_size):
    self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    self.state_size = state_size
    self.action_size = action_size
    #Q-learning
    #two networks -> local q network and target q network
    self.local_qnetwork = Network(state_size,action_size).to(self.device)
    self.target_qnetwork = Network(state_size,action_size).to(self.device)
    #optimizer
    #parameters -> weights
    self.optimizer = optim.Adam(self.local_qnetwork.parameters(),lr = learning_rate)
    #memeory of AI
    self.memory = ReplayMemory(replay_buffer_size)
    #timestamp ->temp step counter to decide at which moment we are gonna learn and update the network
    self.t_stamp = 0

  #stores experiences and decide when to learn from them
  def step(self,state,action,reward,next_state,done):
    #store exp in replay memory
    self.memory.push((state,action,reward,next_state,done))
    #we want to learn every 4 steps.
    #inc. the t_stamp , % will return the remaining
    self.t_stamp = (self.t_stamp + 1)%4
    if self.t_stamp == 0:
      if len(self.memory.memory) > minibatch_size:
        experiences = self.memory.sample(100)
        self.learn(experiences, discount_factor)

  #will select an action based on the given state in the env
  #we are doing epsilon greedy action selection policy
  def act(self, state, epsilon = 0.):
    #state is currently numpy array make it torch tesor so. that the nn can learn from that
    #unsqueeze(0) -> we need to add extra dimension to our state for deep learning vector which will correspond to batch,0->added at the beginning
    state = torch.from_numpy(state).float().unsqueeze(0).to(self.device)
    #forward past this state to q networks to get the action values, 1st we need to set q network to eval mode
    self.local_qnetwork.eval()
    #check to make sure we are not in trainig mode but inference mode making prediction
    with torch.no_grad():
      action_values = self.local_qnetwork(state)
    #back to training mode
    self.local_qnetwork.train()
    if random.random() > epsilon:
      #argmax take numpy data format
      return np.argmax(action_values.cpu().data.numpy())
    else:
      return random.choice(np.arange(self.action_size))

  #updates agents q values based on sampled experiences
  def learn(self,experiences,discount_factor):
    states,next_states,actions,rewards,dones = experiences
    #get max pred Q value for the next states from the traget network
    #detach the action values in tensor to get the max val of them. Detach fun() actually detaches the resulting tensor from the
    #computation graph meaning that we won't be tracking the gradients for this tensor during backward propogation.
    #max(1) -> we need max value along dimension 1 which corresponds to action dimension in the tensor
    #max(1) gives two tensors on for max values and other for indices corresponding to actions for this max values so we use [0]
    next_q_targets = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1) #unsqueezse for batch
    q_targets = rewards + (discount_factor * next_q_targets * (1-dones))
    #current q values , gather all q values
    q_expected = self.local_qnetwork(states).gather(1,actions)
    #loss btm expected and target q values
    loss = F.mse_loss(q_expected,q_targets)
    #backpropogate the loss
    #1st we do reset the optimizer
    self.optimizer.zero_grad()
    loss.backward()
    #single optimization steps lo update the parameters of the model
    self.optimizer.step()
    #update the target network parameters with those of the local network
    self.soft_update(self.local_qnetwork,self.target_qnetwork,interpolation_parameter)


  def soft_update(self,local_model,target_model,interpolation_parameter):
    for target_param,local_param in zip(target_model.parameters(),local_model.parameters()):
      target_param.data.copy_(interpolation_parameter * local_param.data + (1.0 - interpolation_parameter) * target_param.data)

### Initializing the DQN agent

In [ ]:
agent = Agent(state_size,number_actions)

### Training the DQN agent

In [ ]:
#max no. of episodes with which we want to train our agent
number_episodes = 2000
#max no. of time steps per episode
max_t = 1000
#hyperparamenters realted to epsilonn greedy actiom selection policy
eps_start = 1.0
eps_end = 0.01
#decrement for which we are gng from 1.0 to 0.01
eps_decay = 0.995
epsilon = eps_start
#contains scores of last 100 episodes
scores_window = deque(maxlen = 100)

for episode in range(1,number_episodes+1):
  #reset env to initial state
  state, _ = env.reset()
  #initialize the score
  score = 0
  #time steps over the episode
  for t in range(max_t):
    action = agent.act(state,epsilon)
    next_state,reward,done,_,_ = env.step(action)
    #training
    agent.step(state,action,reward,next_state,done)
    state = next_state
    score += reward
    if done:
      break
  scores_window.append(score)
  epsilon = max(eps_end,eps_decay*epsilon)
  #advanced printing - overriding effect
  print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_window)), end="")
  if episode % 100 == 0:
    print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_window)))
    if np.mean(scores_window)>=200.0:
      print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(episode-100, np.mean(scores_window)))
      #save the winning model to file checkpoint.pth check files section
      torch.save(agent.local_qnetwork.state_dict(), 'checkpoint.pth')
      #break the training loop since we reached the solution
      break

Episode 100	Average Score: -155.50
Episode 200	Average Score: -118.36
Episode 300	Average Score: -63.47
Episode 400	Average Score: 21.89
Episode 500	Average Score: 80.00
Episode 600	Average Score: 162.74
Episode 700	Average Score: 178.47
Episode 800	Average Score: 175.67
Episode 900	Average Score: 220.92

Environment solved in 800 episodes!	Average Score: 220.92


## Part 3 - Visualizing the results

In [ ]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env_name):
    env = gym.make(env_name, render_mode='rgb_array')
    state, _ = env.reset()
    done = False
    frames = []
    while not done:
        frame = env.render()
        frames.append(frame)
        action = agent.act(state)
        state, reward, done, _, _ = env.step(action.item())
    env.close()
    imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent, 'LunarLander-v3')

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()